In [1]:
import numpy as np
import pandas as pd

# 1. Load all raw tables
df_clients = pd.read_csv("../data/raw/dim_clients.csv")
df_leads = pd.read_csv("../data/raw/dim_lead_sources.csv")
df_projects = pd.read_csv("../data/raw/fact_projects.csv")
df_costs = pd.read_csv("../data/raw/fact_project_costs.csv")
df_maintenance = pd.read_csv("../data/raw/fact_maintenance.csv")

print("=== RAW RECORD COUNTS ===")
print(f"Clients:        {len(df_clients)}")
print(f"Lead Sources:   {len(df_leads)}")
print(f"Projects:       {len(df_projects)}")
print(f"Cost Records:   {len(df_costs)}")
print(f"Maintenance:    {len(df_maintenance)}")

print("\n" + "=" * 50)
print("=== PROJECTS TABLE AUDIT: DATA TYPES & NULLS ===")
print(df_projects.info())

print("\n" + "=" * 50)
print("=== NUMERICAL DISTRIBUTION CHECK ===")
print(
    df_projects[
        ["area_sqft", "contract_value_aed", "estimated_budget_aed"]
    ].describe()
)

=== RAW RECORD COUNTS ===
Clients:        120
Lead Sources:   4
Projects:       300
Cost Records:   300
Maintenance:    125

=== PROJECTS TABLE AUDIT: DATA TYPES & NULLS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   project_id            300 non-null    object 
 1   client_id             300 non-null    object 
 2   lead_source_id        300 non-null    object 
 3   project_type          300 non-null    object 
 4   area_sqft             300 non-null    int64  
 5   contract_value_aed    300 non-null    int64  
 6   estimated_budget_aed  300 non-null    float64
 7   start_date            300 non-null    object 
 8   planned_end_date      300 non-null    object 
 9   actual_end_date       285 non-null    object 
 10  status                300 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 25.9

In [2]:
import os
import numpy as np
import pandas as pd

# 1. Convert project date strings to real datetime objects
date_columns = ["start_date", "planned_end_date", "actual_end_date"]
for col in date_columns:
    df_projects[col] = pd.to_datetime(df_projects[col])

# 2. Operational Timeline Metrics
df_projects["planned_duration_days"] = (
    df_projects["planned_end_date"] - df_projects["start_date"]
).dt.days

# For completed projects, calculate actual duration and delay days
df_projects["actual_duration_days"] = (
    df_projects["actual_end_date"] - df_projects["start_date"]
).dt.days

df_projects["schedule_delay_days"] = (
    df_projects["actual_end_date"] - df_projects["planned_end_date"]
).dt.days

# Fill missing delay days for "In Progress" projects with 0 for clean aggregation
df_projects["schedule_delay_days_filled"] = df_projects[
    "schedule_delay_days"
].fillna(0)
df_projects["is_delayed"] = (df_projects["schedule_delay_days"] > 0).astype(int)

# 3. Financial Integration: Merge Projects with Costs
df_projects_financials = df_projects.merge(df_costs, on="project_id", how="left")

# Profit & Margin Calculations
df_projects_financials["gross_profit_aed"] = (
    df_projects_financials["contract_value_aed"]
    - df_projects_financials["actual_total_cost_aed"]
)

df_projects_financials["gross_margin_pct"] = round(
    (
        df_projects_financials["gross_profit_aed"]
        / df_projects_financials["contract_value_aed"]
    )
    * 100,
    2,
)

# Cost Variance (Overrun) Calculations
df_projects_financials["cost_variance_aed"] = (
    df_projects_financials["actual_total_cost_aed"]
    - df_projects_financials["estimated_budget_aed"]
)

df_projects_financials["cost_overrun_pct"] = round(
    (
        df_projects_financials["cost_variance_aed"]
        / df_projects_financials["estimated_budget_aed"]
    )
    * 100,
    2,
)

# Subcontractor Dependency Ratio
df_projects_financials["subcontractor_pct_of_cost"] = round(
    (
        df_projects_financials["subcontractor_cost_aed"]
        / df_projects_financials["actual_total_cost_aed"]
    )
    * 100,
    2,
)

# 4. Export Cleaned & Transformed Data to processed/
os.makedirs("../data/processed", exist_ok=True)
df_projects_financials.to_csv(
    "../data/processed/clean_projects_financials.csv", index=False
)
df_clients.to_csv("../data/processed/clean_clients.csv", index=False)
df_leads.to_csv("../data/processed/clean_lead_sources.csv", index=False)
df_maintenance.to_csv("../data/processed/clean_maintenance.csv", index=False)

print("Status: Cleaning & Feature Engineering Complete!")
print(
    f"Cleaned dataset saved: clean_projects_financials.csv ({df_projects_financials.shape[1]} columns)"
)

# Quick sanity preview of key calculated fields
df_projects_financials[
    [
        "project_id",
        "contract_value_aed",
        "actual_total_cost_aed",
        "gross_profit_aed",
        "gross_margin_pct",
        "cost_overrun_pct",
        "schedule_delay_days",
    ]
].head()

Status: Cleaning & Feature Engineering Complete!
Cleaned dataset saved: clean_projects_financials.csv (28 columns)


,project_id,contract_value_aed,actual_total_cost_aed,gross_profit_aed,gross_margin_pct,cost_overrun_pct,schedule_delay_days
0,PRJ-2023-001,538000,352900.0,185100.0,34.41,-7.71,-5.0
1,PRJ-2023-002,276500,200900.0,75600.0,27.34,6.13,-3.0
2,PRJ-2023-003,546000,348200.0,197800.0,36.23,-5.82,16.0
3,PRJ-2023-004,550500,410200.0,140300.0,25.49,5.07,26.0
4,PRJ-2023-005,86000,73800.0,12200.0,14.19,29.70,0.0
